# PART I: Creating the Study's Datasets

# 0. Setup

Before we begin, make sure you have installed all the required Python packages. (The instructions below use pip. You can use easy_install, too.) Also, consider using virtualenv for a cleaner installation experience instead of sudo. I also recommend to running the code via IPython Notebook.
* pip install --upgrade turicreate
* pip install --upgrade networkx
* pip install --upgrade pymongo



Please download the KDD Cup 2016 data, and please also download the project files from our GitHub repository. Through this research, we use the various constants that appear in consts.py. Please change the DATASETS_AMINER_DIR, DATASETS_BASE_DIR, and SFRAMES_BASE_DIR to your local directories, where you can download the datasets and save the project's SFrames.

**Note: Creating the following SFrame requires considerable computation power for long periods.** 

In [1]:
%load_ext autoreload
%autoreload 2
%aimport
%matplotlib inline  

Modules to reload:
all-except-skipped

Modules to skip:



In [2]:
import os
import sys
nb_dir = os.path.split(os.getcwd())[0]
if nb_dir not in sys.path:
    sys.path.append(nb_dir)

# 1. Creating the SFrames

In this study, we used the following datasets:
* [The Microsoft Academic KDD Cup 2016 dataset](https://kddcup2016.azurewebsites.net/Data) - The Microsoft Academic KDD Cup Graph dataset (referred to as the MAG 2016 dataset) contains data on over 126 million papers. The main advantage of this dataset is that it has undergone several preprocessing iterations of author entity matching (any author is identified by ID) and paper deduplication. Additionally, the dataset match between papers and their fields of study includes the hierarchical structure and connections between various fields of study.  <br/>  The link is dead...

* [AMiner dataset](https://aminer.org/open-academic-graph) - The AMiner dataset contains information on over 154 million papers collected by the AMiner team. The dataset contains papers' abstracts, ISSNs, ISBNs, and details on each paper. <br/>  Cureently there is a V2 and to download which one...

* [SJR dataset](http://www.scimagojr.com/journalrank.php) -  The SCImago Journal Rank open dataset (referred to as the SJR dataset) contains journals and country specific metric data starting from 1999. In this study, we used the SJR dataset to better understand how various journal metrics have changed over time. <br/>  How to download?


## 1.1 The Microsoft Academic KDD Cup Dataset

The first step is to convert the dataset text files into SFrame objects using the code located under the SFrames creator directory, using the following code.

In [3]:
from ScienceDynamics.datasets.microsoft_academic_graph import MicrosoftAcademicGraph

In [4]:
from ScienceDynamics.config.configs import DATASETS_BASE_DIR
mag = MicrosoftAcademicGraph(DATASETS_BASE_DIR / "MicrosoftAcademicGraph.zip")

In [6]:
from ScienceDynamics.sframe_creators.create_mag_sframes import create_all_sframes
create_all_sframes() # running this can take considerable time

RuntimeError: Directory: /data/tmp does not exist

The above two lines of code will create a set of SFrames with all the dataset data. The SFrames will include data on authors’ papers, keywords, fields of study, and more. Moreover, the code will construct the Extended Papers SFrame, which contains various meta data on each paper in the dataset.

In [9]:
mag._sframe_dir.joinpath("PaperReferences.sframe").exists()

True

In [11]:
mag._sframe_dir.joinpath("PaperReferences.sframe").exists()

True

In [7]:
ref = ref.rename({"PapOriginal venue name reference ID": "Paper reference ID"})

In [12]:
ref.save(str(mag._sframe_dir.joinpath("PaperReferences.sframe")))

In [5]:
ref = mag.references
ref

Paper ID,PapOriginal venue namereference ID ...
0000006C,5E3BDA99
000002FB,781F8D75
00000698,7DB78D20
0000082C,7FF508C2
00000939,7D75C2AB
00000EBA,7A34C7AF
00000EBA,7B74BB2A
000011C0,80CF80C1
00001561,77B3BE94
00001561,7906CD45


In [ ]:
mag_sf = mag.extended_papers
mag_sf

Less than 64 successfully started. Using only 28 workers.

All operations will proceed as normal, but lambda operations will not be able to use all available cores.

Using default 64 lambda workers.

To maximize the degree of parallelism, add the following code to the beginning of the program:

"turicreate.config.set_runtime_config('TURI_DEFAULT_NUM_PYLAMBDA_WORKERS', 128)"

Note that increasing the degree of parallelism also increases the memory footprint.

In [2]:
mag_sf = tc.load_sframe(EXTENDED_PAPERS_SFRAME)
mag_sf

Paper ID,Original paper title,Normalized paper title,Paper publish year,Paper publish date
01B27BE8,Evaluating Polarity forVerbal Phraseological ...,evaluating polarity forverbal phraseological ...,2014,2014/11/16
027D0030,Automatic Monitoring theContent of Audio ...,automatic monitoring thecontent of audio ...,2012,2012/10/27
7CFE299E,Towards a set of Measuresfor Evaluating Software ...,towards a set of measuresfor evaluating software ...,2009,2009/11
59BEBE1C,Learning ProbabilityDensities of Optimiza ...,learning probabilitydensities of optimiza ...,2008,2008/10/27
5873C011,Towards a Model for anImmune System ...,towards a model for animmune system ...,2002,2002/04/22
7A1109E4,Approach Towards aNatural Language Anal ...,approach towards anatural language anal ...,2013,2013/11
0B00AFD8,Towards the creation ofsemantic models based on ...,towards the creation ofsemantic models based on ...,2012,2012/10/27
5C66D743,Comparison of NeuralNetworks and Support ...,comparison of neuralnetworks and support ...,2009,2009/11/01
040121AE,Multiple Kernel SupportVector Machine Proble ...,multiple kernel supportvector machine proble ...,2014,2014/11/16
7DEADC9A,A Set of Test Cases forPerformance Measures in ...,a set of test cases forperformance measures in ...,2008,


In our study, we also analyzed how various authors' attributes, such as the number of published papers, number of coauthors, etc., has changed over time. To achieve this, we created an authors features SFrame using the following code:

In [3]:
from create_mag_authors_sframe import *
a = AuthorsFeaturesExtractor()

#This need to run on a strong server and can take considerable time to run
a_sf = a.get_authors_all_features_sframe()
a_sf #the SFrame can be later loaded using tc.load_sframe(AUTHROS_FEATURES_SFRAME)

Author ID,Papers by Years Dict,Coauthors by Years Dict,Affilation by Year Dict
00001F05,"{2010: ['5DA0F250'],2013: ['7AF8ABFE']} ...","{2013: ['77CE16EC','17B20BAE']} ...","{2010: [''], 2013: ['']}"
00002AD3,"{2009: ['7A0B348F'],2010: ['795F56C6'], 2 ...","{2009: ['7FD1B86A','7921EA7D', '05390F01', ...","{2009: [''], 2010: [''],2012: [''], 2006: [''], ..."
00006A31,{2009: ['7CFAEB15']},"{2009: ['7E24B147','7C3ED158', '79A6FF42', ...",{2009: ['']}
0000B5FA,"{2008: ['7714EB4E'],2009: ['78B7C257'], 2 ...","{2008: ['54648B9B'],2009: ['7ADCCDB0', ...","{2008: [''], 2009: [''],2010: ['', '', ''], 2 ..."
0001CF9B,{2013: ['7C9BBC3A']},"{2013: ['852809D8','77FE1F64', '80B5223D', ...",{2013: ['']}
00040294,"{2009: ['7892886A','81263516', '81424AA7'], ...","{2009: ['75D367F6','82C1C4DE', '80824D21', ...","{2009: ['', '', ''],2011: ['', ''], 2004: ..."
00045553,"{1987: ['77F10A3D'],1988: ['7836E5B8'], 1 ...","{1987: ['85CAEB12'],1988: ['819D7046', ...","{1987: ['new york medicalcollege'], 1988: [''], ..."
0004B8AF,{2010: ['77AFDEB4']},"{2010: ['77227437','77CB65A7', '5EBF97A1', ...",{2010: ['']}
000510E2,"{2011: ['80790612'],2012: ['76E5D7F2', ...","{2011: ['82D84635','11F1B283'], 2012: ...","{2011: ['university ofqueensland'], 2012: ['', ..."
00063841,{2014: ['7790AFD4']},"{2014: ['853EBBF2','7901305E', '0F71473E', ...",{2014: ['']}


The above SFrame contains various features of each author that were constructed based on analyzing the author’s papers that have at least 5 references. If you notice, the author’s SFrame contains each author’s gender prediction. This column was created by obtaining first-name gender statistics from the[SSA Baby Names](http://www.ssa.gov/oact/babynames/names.zip]) and [WikiTree](https://www.wikitree.com/wiki/Help:Database_Dumps) datasets which include over 115 thousands unique first names (see details in geneder_classifier.py). 

## 1.2 The AMiner Dataset

After downloading the [AMiner website](https://aminer.org/open-academic-graph), simply load to an SFrame using the following code:

In [4]:
aminer_sf = tc.SFrame.read_json('%s/*.txt' % AMINER_DATA_DIR,  orient='lines')
aminer_sf # the SFrame can be accessed also by using tc.load_sframe(AMINER_PAPERS_SFRAME)

abstract,authors,doi,id
None,"[{'name': 'G. Adam'},{'name': 'K. Schreibe ...",10.1002/ange.19650770204,53e99784b7602d9701f3e130
None,"[{'name': 'R. Farahbod'},{'name': 'V. Gervasi'}, ...",None,53e99784b7602d9701f3e131
The method to makingtechnology roadmap is ...,"[{'name': 'MO Chou'},{'name': 'CHEN Jiqing'}, ...",None,53e99784b7602d9701f3e132
Drought is the firstplace in all the natural ...,"[{'name': 'PeijuanWang'}, {'name': 'Jiahua ...",10.1109/IGARSS.2011.6049503 ...,53e99784b7602d9701f3e133
Determination of totalsugar can serve to ...,[{'org': 'YantaiInstitute of Coastal ...,None,53e99784b7602d9701f3e135
Resumen: Uno de losproblemas que debemos ...,[{'name': 'CELSOVARGAS'}] ...,None,53e99784b7602d9701f3e136
None,"[{'name': 'D J Lum'},{'name': 'V Upadhyay'}, ...",10.1111/j.1365-2559.2007.02817.x ...,53e99784b7602d9701f3e137
This paper discussed theplanning and design ...,[{'org': 'School ofResource and ...,None,53e99784b7602d9701f3e139
Rough set is amathematical tool to ...,None,None,53e99784b7602d9701f3e13a
None,"[{'name': 'FTHOUVENYPAISANT'}, ...",10.1016/S0221-0363(05)76274-0 ...,53e99784b7602d9701f3e13b


## 1.3 The SJR Dataset

First, we download all the journal ranking files from [the SJR website](http://www.scimagojr.com/journalrank.php).
Next, we use the following code to create a single SFrame with all the journal data:

In [5]:
from create_sjr_sframe import *
sjr_sf = create_sjr_sframe(SJR_FILES_DIR)
sjr_sf # the SFrame can also be accessed using tc.load_sframe(SJR_SFRAME)

Rank,Title,Type,SJR,SJR Best Quartile,H index,Total Docs.,Total Docs. (3years)
1,Astrophysical JournalLetters ...,journal,61.473,Q1,82,5,7
1,Astrophysical JournalLetters ...,journal,61.473,Q1,82,5,7
2,Annual Review ofBiochemistry ...,journal,49.476,Q1,248,30,81
2,Annual Review ofBiochemistry ...,journal,49.476,Q1,248,30,81
3,Cell,journal,41.978,Q1,616,354,1359
3,Cell,journal,41.978,Q1,616,354,1359
4,Annual Review ofImmunology ...,journal,40.906,Q1,254,29,81
4,Annual Review ofImmunology ...,journal,40.906,Q1,254,29,81
5,Annual Review of Cell andDevelopmental Biology ...,book serie,33.882,Q1,182,25,61
5,Annual Review of Cell andDevelopmental Biology ...,book serie,33.882,Q1,182,25,61


## 1.4 Joint Datasets

The MAG and AMiner datasets have a slightly different set of features. While the MAG dataset contains data on each author with a unique author ID, the AMiner contains additional data on each paper, including the paper's abstract and the paper's ISSN or ISBN. Additionally, the SJR dataset contains data about each journal's ranking.

To combine the data from the author publication record and the journals' rankings, we join the datasets. First, we joined the MAG and AMiner datasets by matching DOI values, using the following code (see also create_mag_aminer_sframe.py):

In [6]:
sf = tc.load_sframe(EXTENDED_PAPERS_SFRAME)
g1 = sf.groupby('Paper Document Object Identifier (DOI)', {'Count': agg.COUNT()})
s1 = set(g1[g1['Count'] > 1]['Paper Document Object Identifier (DOI)'])
sf = sf[sf['Paper Document Object Identifier (DOI)'].apply(lambda doi: doi not in s1 )]
sf.materialize()

sf2 = tc.load_sframe(AMINER_PAPERS_SFRAME)
g2 = sf2.groupby('doi', {'Count': agg.COUNT()})
s2 = set(g2[g2['Count'] > 1]['doi'])
sf2 = sf2[sf2['doi'].apply(lambda doi: doi not in s2 )]
sf2.materialize()

aminer_mag_sf = sf.join(sf2, {'Paper Document Object Identifier (DOI)': 'doi'})
aminer_mag_sf['title_len'] = aminer_mag_sf['title'].apply(lambda t: len(t))
aminer_mag_sf = aminer_mag_sf[aminer_mag_sf['title_len'] > 0]
aminer_mag_sf = aminer_mag_sf.rename({"Paper ID": "MAG Paper ID", "id": "Aminer Paper ID"})
aminer_mag_sf.remove_column('title_len')
aminer_mag_sf # this SFrame can be accessed using tc.load_Sframe(AMINER_MAG_JOIN_SFRAME)

MAG Paper ID,Original paper title,Normalized paper title,Paper publish year,Paper publish date
7C15F682,"Ptychoptera deleta Novak,1877 from the Early ...",ptychoptera deleta novak1877 from the early ...,2011,2011
84A37D36,Sherborn’sforaminiferal studies ...,sherborn s foraminiferalstudies and their ...,2016,2016/07/01
773B216E,A new species ofhydrobiid snails ...,a new species ofhydrobiid snails moll ...,2011,2011/10/19
77C44F83,Revision of theplanthopper genus ...,revision of theplanthopper genus ...,2014,2014/10/12
75233F3E,Female genitalia ofSeasogonia Young from ...,female genitalia ofseasogonia young from ...,2012,2012/11/01
7B5321C5,A taxonomic study on thegenus Ettchellsia ...,a taxonomic study on thegenus ettchellsia cam ...,2012,2012
3C77A5B8,"An Asiatic Chironomid inBrazil: morphology, DNA ...",an asiatic chironomid inbrazil morphology dna ...,2015,2015/07/27
8051111A,A new species ofSmicromorpha ...,a new species ofsmicromorpha hymenoptera ...,2009,2009/09/14
79BD8F37,Open exchange ofscientific knowledge and ...,open exchange ofscientific knowledge and ...,2014,2014/06/06
240B7EFF,"Four new species ofEpicephala Meyrick, 1880 ...",four new species ofepicephala meyrick 1880 ...,2015,2015/06/15


Using the joined dataset, we obtained an SFrame with the joint meta data of 28.9 million papers. We can take this SFrame and join it with the SJR dataset.

In [8]:
import re
def create_aminer_mag_sjr_sframe(year):
    """
    Creates a unified SFrame of AMiner, MAG, and the SJR datasets
    :param year: year to use for SJR data
    :return: SFrame with AMiner, MAG, and SJR data
    :rtype: tc.SFrame
    """
    sf = tc.load_sframe(AMINER_MAG_JOIN_SFRAME)
    sf = sf[sf['issn'] != None]
    sf = sf[sf['issn'] != 'null']
    sf.materialize()
    r = re.compile("(\d+)-(\d+)")
    sf['issn_str'] = sf['issn'].apply(lambda i: "".join(r.findall(i)[0]) if len(r.findall(i))> 0 else None)
    sf = sf[sf['issn_str'] != None]
    sjr_sf = tc.load_sframe(SJR_SFRAME)
    sjr_sf = sjr_sf[sjr_sf['Year'] == year]
    return sf.join(sjr_sf, on={'issn_str': "ISSN"})
create_aminer_mag_sjr_sframe(2015)

MAG Paper ID,Original paper title,Normalized paper title,Paper publish year,Paper publish date
7C15F682,"Ptychoptera deleta Novak,1877 from the Early ...",ptychoptera deleta novak1877 from the early ...,2011,2011
773B216E,A new species ofhydrobiid snails ...,a new species ofhydrobiid snails moll ...,2011,2011/10/19
77C44F83,Revision of theplanthopper genus ...,revision of theplanthopper genus ...,2014,2014/10/12
75233F3E,Female genitalia ofSeasogonia Young from ...,female genitalia ofseasogonia young from ...,2012,2012/11/01
7B5321C5,A taxonomic study on thegenus Ettchellsia ...,a taxonomic study on thegenus ettchellsia cam ...,2012,2012
79BD8F37,Open exchange ofscientific knowledge and ...,open exchange ofscientific knowledge and ...,2014,2014/06/06
778DE072,"First report onC-banding, fluorochrome ...",first report on c bandingfluorochrome staining ...,2013,2013/07/30
76034B62,"Checklist of the familiesScathophagidae, Fanni ...",checklist of the familiesscathophagidae fanniidae ...,2014,2014/09/19
781099DE,Checklist of the familyCulicidae (Diptera) in ...,checklist of the familyculicidae diptera in ...,2014,2014
770A539F,"Two new species ofharvestmen (Opiliones, ...",two new species ofharvestmen opiliones ...,2014,2014/08/14


# 2. Loading the Dataset to MongoDB

Using Turicreate and SFrame objects can help us get general data on how academic publication dynamics have changed over time, but it would be challenging to use this data to create more complicated insights, such as the trends of a specific journal. To reveal more complicated insights using the data, we would need to load the dataset to a different framework. In this study, we chose to use MongoDB as our framework for more complicated queries.
We installed MongoDB on Ubuntu 17.10 using the instructions in the following [link](https://medium.com/gatemill/how-to-install-mongodb-3-6-on-ubuntu-17-10-ac0bc225e648). After MongoDB is installed and running, please remember to set the user and password, and update MONGO_HOST & MONGO_PORT vars in consts.py (one can also adjust the connection to include user password auth).
Now, the next step is to load the above created SFrames to collections in MongoDB using mongo_connecter.py:

In [9]:
from mongo_connector import load_sframes
load_sframes() #this will load the SFrame to a local

In the end of the loading process, six collections will be loaded to the journal database.

In [10]:
MD.client.journals.collection_names()

[u'authoros_features',
 u'sjr_journals',
 u'aminer_mag_papers',
 u'fields_of_study_papers',
 u'papers_features',
 u'authors_features']

In the second part of the tutorial, we will demonstrate how the above created MongoDB collections can be utilized to calculate various statistics on paper collections, authors, journals, and research domains.